# Bioacoustic recording download — fire-prone LGAs

Pulls as much historical audio recording metadata (and optionally the audio files themselves) as possible for every local government area listed in `fire_prone_divisions.csv`, using the LGA polygons in `au_admin2_boundaries.geojson` to build per-LGA search queries.

**Source: Xeno-canto.** ALA (`galah`, used in `animal_sightings.ipynb`) was tried first since it's the same pipeline already set up for this project, but the installed `galah` build has a bug in `atlas_media()`: it always builds its metadata request from the occurrence's `images` column, even when only `multimedia=["sounds"]` is requested, and throws/500s on sound-only records. Xeno-canto's public API has a clean bounding-box search and direct download URLs per recording, plus much deeper historical depth for bird calls (some recordings go back to the 1950s-60s), so it's the more reliable option here.

**Before running:** register a free account at https://xeno-canto.org and copy your API key from https://xeno-canto.org/account, then paste it into the config cell below.

In [ ]:
import json
import time
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import Point

In [ ]:
XC_API_KEY = "PASTE_YOUR_XENO_CANTO_KEY_HERE"  # https://xeno-canto.org/account
XC_BASE_URL = "https://xeno-canto.org/api/3/recordings"

FIRE_PRONE_CSV = "fire_prone_divisions.csv"
BOUNDARIES_GEOJSON = "au_admin2_boundaries.geojson"

METADATA_OUT = "xeno_canto_recordings_metadata.csv"
AUDIO_DIR = Path("xc_audio")

REQUEST_SLEEP_SECONDS = 1.0  # be polite to the API between requests

## 1. Match fire-prone LGAs to their boundary polygons

In [ ]:
fire_prone = pd.read_csv(FIRE_PRONE_CSV)
boundaries = gpd.read_file(BOUNDARIES_GEOJSON)

lgas = boundaries.merge(
    fire_prone[["ADM1_NAME", "ADM2_NAME", "fire_count", "earliest_fire", "latest_fire"]],
    on=["ADM1_NAME", "ADM2_NAME"],
    how="inner",
)
print(f"Matched {len(lgas)} of {len(fire_prone)} fire-prone divisions to boundary polygons")

missing = set(zip(fire_prone.ADM1_NAME, fire_prone.ADM2_NAME)) - set(zip(lgas.ADM1_NAME, lgas.ADM2_NAME))
if missing:
    print("No boundary match for:", missing)

lgas.head()

## 2. Xeno-canto query helpers

`box:` search is a bounding-box query (south,west,north,east — i.e. lat_min,lon_min,lat_max,lon_max) so results are then filtered down to points that actually fall inside the LGA polygon, not just its bounding box. No date/year filter is applied to the query itself, so each LGA returns its full available historical range.

In [ ]:
def xc_box_query(geometry):
    lon_min, lat_min, lon_max, lat_max = geometry.bounds
    return f"box:{lat_min},{lon_min},{lat_max},{lon_max}"


def fetch_xc_recordings(box_query, api_key, sleep_seconds=REQUEST_SLEEP_SECONDS):
    """Pull every page of xeno-canto recordings matching a query."""
    recordings = []
    page = 1
    num_pages = 1
    while page <= num_pages:
        resp = requests.get(
            XC_BASE_URL,
            params={"query": box_query, "key": api_key, "page": page},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        if "error" in data:
            raise RuntimeError(f"Xeno-canto API error: {data}")
        recordings.extend(data.get("recordings", []))
        num_pages = data.get("numPages", 1)
        page += 1
        if page <= num_pages:
            time.sleep(sleep_seconds)
    return recordings

## 3. Sanity check on one LGA

Run this once `XC_API_KEY` is set to confirm the query works and to inspect the real field names xeno-canto returns (field names occasionally drift between API versions).

In [ ]:
sample = lgas.iloc[0]
sample_query = xc_box_query(sample.geometry)
print(sample.ADM1_NAME, "/", sample.ADM2_NAME, "->", sample_query)

sample_recordings = fetch_xc_recordings(sample_query, XC_API_KEY)
print(f"{len(sample_recordings)} recordings found in the bounding box")
if sample_recordings:
    print(json.dumps(sample_recordings[0], indent=2))

## 4. Pull metadata for every fire-prone LGA

This is the full run across all 67 divisions. It queries each LGA's bounding box, keeps only recordings whose coordinates fall inside the actual polygon, and tags each recording with the LGA and fire-history columns from `fire_prone_divisions.csv`.

In [ ]:
all_records = []

for i, row in lgas.iterrows():
    box_query = xc_box_query(row.geometry)
    try:
        recs = fetch_xc_recordings(box_query, XC_API_KEY)
    except Exception as exc:
        print(f"  FAILED {row.ADM1_NAME} / {row.ADM2_NAME}: {exc}")
        continue

    kept = 0
    for rec in recs:
        try:
            lat, lon = float(rec["lat"]), float(rec["lng"])
        except (TypeError, ValueError, KeyError):
            continue
        if not row.geometry.contains(Point(lon, lat)):
            continue  # bbox hit that falls outside the actual LGA polygon
        rec = dict(rec)
        rec["ADM1_NAME"] = row.ADM1_NAME
        rec["ADM2_NAME"] = row.ADM2_NAME
        all_records.append(rec)
        kept += 1

    print(f"{row.ADM1_NAME} / {row.ADM2_NAME}: {kept} inside polygon ({len(recs)} in bbox)")
    time.sleep(REQUEST_SLEEP_SECONDS)

recordings_df = pd.DataFrame(all_records)
recordings_df.to_csv(METADATA_OUT, index=False)
print(f"Saved {len(recordings_df)} recordings across {lgas.shape[0]} LGAs to {METADATA_OUT}")

## 5. Historical coverage summary

In [ ]:
dates = pd.to_datetime(recordings_df["date"], errors="coerce")
print("Earliest recording:", dates.min())
print("Latest recording:", dates.max())
print("Total recordings:", len(recordings_df))

recordings_df.groupby("ADM2_NAME").size().sort_values(ascending=False).head(20)

## 6. Download the audio files

Off by default — review the metadata CSV first (recording counts and file sizes can add up fast across 67 LGAs). Flip `DOWNLOAD_AUDIO = True` when ready, optionally capping `MAX_FILES_PER_LGA` for a first test pass.

In [ ]:
DOWNLOAD_AUDIO = False
MAX_FILES_PER_LGA = None  # e.g. set to 10 to test before pulling everything


def download_recording(rec, dest_dir):
    file_url = rec.get("file")
    if not file_url:
        return None
    if file_url.startswith("//"):
        file_url = "https:" + file_url
    ext = (rec.get("file-name") or "").rsplit(".", 1)[-1] or "mp3"
    dest_path = dest_dir / f"{rec['id']}.{ext}"
    if dest_path.exists():
        return dest_path
    resp = requests.get(file_url, timeout=60)
    resp.raise_for_status()
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path.write_bytes(resp.content)
    return dest_path


if DOWNLOAD_AUDIO:
    for (adm1, adm2), group in recordings_df.groupby(["ADM1_NAME", "ADM2_NAME"]):
        dest_dir = AUDIO_DIR / adm1.replace(" ", "_") / adm2.replace(" ", "_")
        subset = group if MAX_FILES_PER_LGA is None else group.head(MAX_FILES_PER_LGA)
        downloaded = 0
        for _, rec in subset.iterrows():
            try:
                if download_recording(rec.to_dict(), dest_dir):
                    downloaded += 1
            except Exception as exc:
                print(f"  failed {rec.get('id')}: {exc}")
        time.sleep(REQUEST_SLEEP_SECONDS)
        print(f"Downloaded {downloaded} files -> {dest_dir}")
else:
    print("DOWNLOAD_AUDIO is False — only metadata was saved. Set it to True to pull audio files.")